In [ ]:
import pandas as pd
import requests
import datetime
import time
import numpy as np

# --- CONFIGURATION ---

# API Configuration
# REPLACE THIS with your actual API key
API_KEY = "YOUR_OPENWEATHERMAP_API_KEY" 

# Target Coordinates (Seixal area)
LAT = 38.625833
LON = -9.086561

# Target Coordinates (Lisbon area)
# LAT_LISBON = 38.7557335
# LON_LISBON = -9.1582073
    
# File Paths
INPUT_FILE = 'aggregated_data.csv'
OUTPUT_FILE = 'aggregated_data_enriched.csv'

# API Endpoint Template
BASE_URL = "https://api.openweathermap.org/data/3.0/onecall/timemachine"

In [ ]:
def iso_to_rounded_epoch(iso_str: str) -> int:
    """
    Converts ISO-8601 timestamp to a Unix epoch rounded to the nearest hour.
    Example: '2024-08-19T02:56:00Z' -> rounded epoch for 03:00:00
    """
    try:
        # Handle 'Z' if present for UTC
        dt = datetime.datetime.fromisoformat(iso_str.replace("Z", "+00:00"))
        timestamp = dt.timestamp()
        
        # Round to nearest hour (3600 seconds)
        hours = timestamp / 3600
        rounded_timestamp = round(hours) * 3600
        return int(rounded_timestamp)
    except Exception as e:
        print(f"Error converting timestamp {iso_str}: {e}")
        return None

def get_historical_weather(epoch: int, lat: float, lon: float, api_key: str):
    """
    Fetches historical weather data for a specific epoch time.
    """
    params = {
        'lat': lat,
        'lon': lon,
        'dt': epoch,
        'appid': api_key,
        'units': 'metric'
    }
    
    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status() # Raise error for bad status codes (4xx, 5xx)
        data = response.json()
        
        # Extract the relevant metric from the first data point
        if 'data' in data and len(data['data']) > 0:
            current = data['data'][0]
            return {
                'temperature': current.get('temp'),
                'humidity': current.get('humidity'),
                'pressure': current.get('pressure')
            }
    except requests.exceptions.RequestException as e:
        print(f"API Request failed for epoch {epoch}: {e}")
    except (KeyError, IndexError):
        print(f"Unexpected data format for epoch {epoch}")
        
    return None

In [ ]:
# Load the dataset
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded {len(df)} rows from {INPUT_FILE}")
except FileNotFoundError:
    print(f"Error: File {INPUT_FILE} not found.")
    # Create dummy data for demonstration if file missing
    df = pd.DataFrame() 

# Initialize columns if they don't exist
required_columns = ["weather", "temperature", "humidity", "pressure"]
for col in required_columns:
    if col not in df.columns:
        df[col] = np.nan


In [ ]:
# Filter rows that need processing:
# 1. Environment is 'household'
# 2. 'weather' column is NaN (to avoid re-fetching data we already have)
rows_to_process = df[
    (df["environment"] == "household") & 
    (df["weather"].isna())
].index

print(f"Found {len(rows_to_process)} rows to process.")

for idx in rows_to_process:
    timestamp_str = df.loc[idx, "timestamp"]
    
    # 1. Convert Timestamp
    epoch = iso_to_rounded_epoch(timestamp_str)
    
    if epoch:
        print(f"Processing Index {idx} | Epoch: {epoch}...", end=" ")
        
        # 2. Fetch Data
        weather_data = get_historical_weather(epoch, LAT, LON, API_KEY)
        
        # 3. Update DataFrame
        if weather_data:
            df.loc[idx, "temperature"] = weather_data['temperature']
            df.loc[idx, "humidity"] = weather_data['humidity']
            df.loc[idx, "pressure"] = weather_data['pressure']
            df.loc[idx, "weather"] = 1 # Mark as processed
            print("Success.")
        else:
            print("Failed to fetch data.")
            
        # Optional: Sleep briefly to avoid hitting API rate limits
        # time.sleep(0.5) 
    else:
        print(f"Skipping Index {idx}: Invalid timestamp.")

print("Processing complete.")

In [ ]:
# Check a sample of the processed data
print(df[df["environment"] == "household"][["timestamp", "temperature", "humidity", "pressure"]].head())

# Save to CSV
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved enriched data to {OUTPUT_FILE}")